In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.dataset import load_processed_metadata, JumpSequenceDataset
from src.temporal_model import GRUClassifier
from src.train_temporal import (
    run_single_split_experiment,
    run_stratified_kfold_cv,
    CV_RESULTS_PATH,
    CV_PREDICTIONS_PATH,
    CV_SUMMARY_PATH,
    CV_CONFUSION_MATRIX_PATH,
)
from src.config import INDEX_TO_LABEL

In [2]:
processed_meta = load_processed_metadata()

print("Number of processed clips:", len(processed_meta))
processed_meta.head()

Number of processed clips: 30


,clip_id,video_path,label,sequence_path,num_sampled_frames,missing_pose_frames,processed_shape
0,bad_jump_01,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,0,"(30, 132)"
1,bad_jump_02,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,0,"(30, 132)"
2,bad_jump_03,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,0,"(30, 132)"
3,bad_jump_04,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,1,"(30, 132)"
4,bad_jump_05,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,0,"(30, 132)"


In [3]:
print(processed_meta["label"].value_counts())

label
bad_jump     15
good_jump    15
Name: count, dtype: int64


In [4]:
dataset = JumpSequenceDataset(processed_meta)

print("Dataset length:", len(dataset))

x, y, clip_id = dataset[0]
print("Sequence tensor shape:", x.shape)
print("Label tensor:", y)
print("Clip ID:", clip_id)

Dataset length: 30
Sequence tensor shape: torch.Size([30, 132])
Label tensor: tensor(0)
Clip ID: bad_jump_01


In [5]:
model = GRUClassifier(
    input_dim=132,
    hidden_dim=64,
    num_layers=1,
    num_classes=2,
)

model

GRUClassifier(
  (gru): GRU(132, 64, batch_first=True)
  (classifier): Linear(in_features=64, out_features=2, bias=True)
)

In [6]:
sample_batch = torch.stack([dataset[0][0], dataset[1][0]], dim=0)

print("Sample batch shape:", sample_batch.shape)

logits = model(sample_batch)
print("Logits shape:", logits.shape)
print(logits)

Sample batch shape: torch.Size([2, 30, 132])
Logits shape: torch.Size([2, 2])
tensor([[0.0919, 0.3483],
        [0.4799, 0.0491]], grad_fn=<AddmmBackward0>)


In [7]:
criterion = torch.nn.CrossEntropyLoss()

sample_labels = torch.tensor([dataset[0][1].item(), dataset[1][1].item()])
loss = criterion(logits, sample_labels)

print("Sample labels:", sample_labels)
print("Loss:", loss.item())

Sample labels: tensor([0, 0])
Loss: 0.6651453971862793


In [8]:
from sklearn.model_selection import train_test_split

In [9]:
train_df, val_df = train_test_split(
    processed_meta,
    test_size=0.2,
    stratify=processed_meta["label"],
    random_state=42,
)

print("Train size:", len(train_df))
print("Val size:", len(val_df))

Train size: 24
Val size: 6


Single-split sanity check only. This section verifies that the GRU training loop runs end-to-end, but it is not the primary evaluation because the dataset is too small for a stable single holdout result.

In [10]:
single_split_result = run_single_split_experiment(
    train_df=train_df,
    val_df=val_df,
    num_epochs=5,
    batch_size=4,
    learning_rate=1e-3,
    hidden_dim=64,
)

Epoch 01 | Train Loss: 0.7354 | Train Acc: 0.4583 | Val Loss: 0.7557 | Val Acc: 0.0000
Epoch 02 | Train Loss: 0.7107 | Train Acc: 0.5000 | Val Loss: 0.7644 | Val Acc: 0.5000
Epoch 03 | Train Loss: 0.6545 | Train Acc: 0.6250 | Val Loss: 0.7458 | Val Acc: 0.1667
Epoch 04 | Train Loss: 0.6375 | Train Acc: 0.6250 | Val Loss: 0.7424 | Val Acc: 0.3333
Epoch 05 | Train Loss: 0.6180 | Train Acc: 0.6250 | Val Loss: 0.7410 | Val Acc: 0.1667


In [11]:
single_split_result["history"]

,epoch,train_loss,train_acc,val_loss,val_accuracy,val_precision,val_recall,val_f1
0,1,0.735353,0.458333,0.755654,0.000000,0.00,0.000000,0.000000
1,2,0.710656,0.500000,0.764428,0.500000,0.50,1.000000,0.666667
2,3,0.654455,0.625000,0.745777,0.166667,0.25,0.333333,0.285714
3,4,0.637520,0.625000,0.742422,0.333333,0.00,0.000000,0.000000
4,5,0.618043,0.625000,0.741004,0.166667,0.25,0.333333,0.285714


In [12]:
single_split_result["final_val_metrics"]

{'loss': 0.7410037318865458,
 'accuracy': 0.16666666666666666,
 'precision': 0.25,
 'recall': 0.3333333333333333,
 'f1': 0.2857142857142857,
 'confusion_matrix': array([[0, 3],
        [2, 1]]),
 'labels': [1, 0, 1, 0, 1, 0],
 'preds': [1, 1, 0, 1, 0, 1],
 'clip_ids': ['good_jump_08',
  'bad_jump_07',
  'good_jump_15',
  'bad_jump_04',
  'good_jump_03',
  'bad_jump_13']}

Primary preliminary evaluation: stratified 5-fold cross-validation at the clip level.

In [13]:
cv_outputs = run_stratified_kfold_cv(
    metadata_df=processed_meta,
    n_splits=5,
    num_epochs=5,
    batch_size=4,
    learning_rate=1e-3,
    hidden_dim=64,
    save_results=True,
)


Starting fold 1/5
Epoch 01 | Train Loss: 0.7331 | Train Acc: 0.3750 | Val Loss: 0.7005 | Val Acc: 0.5000
Epoch 02 | Train Loss: 0.6816 | Train Acc: 0.6667 | Val Loss: 0.7243 | Val Acc: 0.3333
Epoch 03 | Train Loss: 0.6679 | Train Acc: 0.6250 | Val Loss: 0.7378 | Val Acc: 0.3333
Epoch 04 | Train Loss: 0.6392 | Train Acc: 0.6667 | Val Loss: 0.7346 | Val Acc: 0.3333
Epoch 05 | Train Loss: 0.6309 | Train Acc: 0.6250 | Val Loss: 0.7363 | Val Acc: 0.3333

Starting fold 2/5
Epoch 01 | Train Loss: 0.7641 | Train Acc: 0.4167 | Val Loss: 0.6912 | Val Acc: 0.5000
Epoch 02 | Train Loss: 0.6829 | Train Acc: 0.5417 | Val Loss: 0.6748 | Val Acc: 0.6667
Epoch 03 | Train Loss: 0.6677 | Train Acc: 0.7083 | Val Loss: 0.6650 | Val Acc: 0.8333
Epoch 04 | Train Loss: 0.6602 | Train Acc: 0.5833 | Val Loss: 0.6622 | Val Acc: 0.5000
Epoch 05 | Train Loss: 0.6497 | Train Acc: 0.5833 | Val Loss: 0.6489 | Val Acc: 0.6667

Starting fold 3/5
Epoch 01 | Train Loss: 0.7984 | Train Acc: 0.5000 | Val Loss: 0.7143 | Va

In [14]:
cv_outputs["fold_results"]

,fold,seed,val_loss,accuracy,precision,recall,f1
0,1,43,0.736258,0.333333,0.333333,0.333333,0.333333
1,2,44,0.648860,0.666667,1.000000,0.333333,0.500000
2,3,45,0.663542,0.666667,0.666667,0.666667,0.666667
3,4,46,0.985435,0.166667,0.250000,0.333333,0.285714
4,5,47,0.669156,0.666667,0.600000,1.000000,0.750000


In [15]:
cv_outputs["summary"]

,metric,mean,std
0,val_loss,0.740650,0.140909
1,accuracy,0.500000,0.235702
2,precision,0.570000,0.297303
3,recall,0.533333,0.298142
4,f1,0.507143,0.202311


In [16]:
cv_outputs["confusion_matrix"]

,pred_0,pred_1
true_0,7,8
true_1,7,8


In [17]:
pred_df = cv_outputs["predictions"].copy()
pred_df["true_label_name"] = pred_df["true_label"].map(INDEX_TO_LABEL)
pred_df["pred_label_name"] = pred_df["pred_label"].map(INDEX_TO_LABEL)

pred_df.head(10)

,fold,clip_id,true_label,pred_label,correct,true_label_name,pred_label_name
0,1,bad_jump_03,0,1,0,bad_jump,good_jump
1,1,bad_jump_07,0,1,0,bad_jump,good_jump
2,1,bad_jump_08,0,0,1,bad_jump,bad_jump
3,1,good_jump_03,1,0,0,good_jump,bad_jump
4,1,good_jump_10,1,1,1,good_jump,good_jump
5,1,good_jump_13,1,0,0,good_jump,bad_jump
6,2,bad_jump_05,0,0,1,bad_jump,bad_jump
7,2,bad_jump_10,0,0,1,bad_jump,bad_jump
8,2,bad_jump_14,0,0,1,bad_jump,bad_jump
9,2,good_jump_08,1,0,0,good_jump,bad_jump


In [18]:
pred_df[pred_df["correct"] == 0]

,fold,clip_id,true_label,pred_label,correct,true_label_name,pred_label_name
0,1,bad_jump_03,0,1,0,bad_jump,good_jump
1,1,bad_jump_07,0,1,0,bad_jump,good_jump
3,1,good_jump_03,1,0,0,good_jump,bad_jump
5,1,good_jump_13,1,0,0,good_jump,bad_jump
9,2,good_jump_08,1,0,0,good_jump,bad_jump
11,2,good_jump_12,1,0,0,good_jump,bad_jump
13,3,bad_jump_11,0,1,0,bad_jump,good_jump
17,3,good_jump_15,1,0,0,good_jump,bad_jump
18,4,bad_jump_01,0,1,0,bad_jump,good_jump
19,4,bad_jump_02,0,1,0,bad_jump,good_jump


In [19]:
print(CV_RESULTS_PATH, CV_RESULTS_PATH.exists())
print(CV_PREDICTIONS_PATH, CV_PREDICTIONS_PATH.exists())
print(CV_SUMMARY_PATH, CV_SUMMARY_PATH.exists())
print(CV_CONFUSION_MATRIX_PATH, CV_CONFUSION_MATRIX_PATH.exists())

/Users/arjunrammohandas/Desktop/JumpSafe_Continuation/data/processed/cv_results.csv True
/Users/arjunrammohandas/Desktop/JumpSafe_Continuation/data/processed/cv_predictions.csv True
/Users/arjunrammohandas/Desktop/JumpSafe_Continuation/data/processed/cv_summary.csv True
/Users/arjunrammohandas/Desktop/JumpSafe_Continuation/data/processed/cv_confusion_matrix.npy True


In [20]:
fold_results = cv_outputs["fold_results"]
metric_cols = ["val_loss", "accuracy", "precision", "recall", "f1"]

print("Mean metrics:")
print(fold_results[metric_cols].mean())

print("\nStd metrics:")
print(fold_results[metric_cols].std())

Mean metrics:
val_loss     0.740650
accuracy     0.500000
precision    0.570000
recall       0.533333
f1           0.507143
dtype: float64

Std metrics:
val_loss     0.140909
accuracy     0.235702
precision    0.297303
recall       0.298142
f1           0.202311
dtype: float64
